# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes directly
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and fields with their `@id`s.

In [ ]:
# List all RecordSets with their @id
print("Available RecordSets:")
for rset in dataset.metadata.record_sets():
    print(f"@id: {rset.id} | name: {rset.name}")
    print("  Fields:")
    for field in rset.fields:
        print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print()

# For this dataset, let's collect record set IDs to be used in the next steps
record_set_ids = [rset.id for rset in dataset.metadata.record_sets()]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}

print("Loading records from available RecordSets...\n")
for rset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rset_id))
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded {len(df)} records from RecordSet: {rset_id}")
        print(f"  Fields: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records from {rset_id}: {e}")

# Choose the first RecordSet for demonstration
if record_set_ids:
    chosen_record_set = record_set_ids[0]
    print(f"\nPreview from {chosen_record_set}:")
    display(dataframes[chosen_record_set].head())
else:
    chosen_record_set = None
    print("No RecordSets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

if chosen_record_set is not None:
    df = dataframes[chosen_record_set].copy()
    print(f"Columns: {df.columns.tolist()}")
    # Try to guess a numeric field: choose the first float/int column
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is not None:
        print(f"Using '{numeric_field}' as numeric_field for demonstration.")
        # Automatic threshold using mean
        if not df[numeric_field].isnull().all():
            threshold = df[numeric_field].mean()
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records")

            # Normalize
            filtered_df[f"{numeric_field}_normalized"] = (
                filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized '{numeric_field}':")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
            # Try grouping by a categorical column
            group_field = None
            for col in df.columns:
                if col != numeric_field and df[col].nunique() < len(df)/5:
                    group_field = col
                    break
            if group_field:
                print(f"Grouping by '{group_field}':\n")
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                display(grouped_df.head())
            else:
                print("No suitable group_field found for demonstration.")
        else:
            print(f"No non-null values found in numeric field {numeric_field}.")
    else:
        print("No numeric field found in the chosen record set.")
else:
    print("No record set chosen due to previous error.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set is not None and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # If group_field exists, boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, you learned how to use the `mlcroissant` library to load and explore the FAIR\^2 dataset on protein abundance in extracellular vesicles from human mast cells. You reviewed its record sets, loaded sample data, performed filtering and normalization on a numeric field, and visualized basic data distributions. The structure and scientific richness of the dataset enable advanced bioinformatics, statistics, and machine learning workflows.*